# **Cell 1 — Cài thư viện**

In [ ]:
!pip install deep-translator underthesea --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.2 MB/s eta 0:00:00


# **Cell 2 — Upload file**

In [ ]:
from google.colab import files
import io, pandas as pd

print('Chọn file: ')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Upload thành công: {len(df)} dòng')
print(df['intent'].value_counts())

Chọn file: 


Saving data_cleaned_with_intent.csv to data_cleaned_with_intent.csv
Upload thành công: 4527 dòng
intent
khen_chat_luong        1835
trung_tinh             1695
dung_voi_mo_ta          269
phan_nan_chat_luong     198
khen_giao_hang          194
gia_tri_gia             143
de_xuat_mua              86
phan_nan_giao_hang       70
khen_dich_vu             31
phan_nan_dich_vu          6
Name: count, dtype: int64


# **Cell 3 — Cấu hình**

In [ ]:
TARGET_ROWS         = 12_000  # Tổng số dòng mục tiêu
MAX_PER_INTENT      = 1_500   # Tối đa sinh thêm mỗi intent
SLEEP_BACKTRANS     = 1.2     # Giây chờ giữa các lần gọi Google Translate
                              # (tránh bị block — KHÔNG giảm xuống dưới 1.0)
OUTPUT_FILE         = 'data_augmented.csv'

# Tỷ lệ kết hợp 2 phương pháp (tổng = 1.0)
RATIO_BACKTRANS     = 0.5     # 50% câu dùng back-translation
RATIO_EDA           = 0.5     # 50% câu dùng EDA

print('Cấu hình sẵn sàng.')

Cấu hình sẵn sàng.


# **Cell 4 — Tính số câu cần sinh thêm cho từng intent**

In [ ]:
def compute_needed(df, target, max_per_intent=1500):
    counts = df['intent'].value_counts().to_dict()
    current_total = sum(counts.values())

    # Đảm bảo target luôn lớn hơn hiện tại
    effective_target = max(target, current_total + len(counts) * 100)
    gap = effective_target - current_total

    if gap <= 0:
        gap = len(counts) * max_per_intent // 2   # Fallback: sinh tối thiểu

    inv_w     = {k: 1/(v+1) for k,v in counts.items()}
    total_inv = sum(inv_w.values())
    needed    = {k: min(round(gap * inv_w[k]/total_inv), max_per_intent)
                 for k in counts}

    # Phân bổ phần dư
    remaining = gap - sum(needed.values())
    if remaining > 0:
        for intent in sorted(needed, key=lambda k: counts.get(k, 0)):
            if needed[intent] < max_per_intent and remaining > 0:
                add = min(remaining, max_per_intent - needed[intent])
                needed[intent] += add
                remaining -= add

    diff = gap - sum(needed.values())
    if diff != 0:
        needed[max(needed, key=needed.get)] += diff

    return needed

# **Cell 5 — Từ điển đồng nghĩa tiếng Việt**

In [ ]:
SYNONYMS = {
    # Khen chất lượng
    'đẹp':          ['xinh', 'xịn', 'oke', 'ngon', 'tuyệt', 'chuẩn', 'ưng'],
    'tốt':          ['ổn', 'ok', 'oke', 'tuyệt', 'chuẩn', 'xịn', 'chắc'],
    'chất lượng':   ['chất_lượng', 'cl', 'chất', 'xịn'],
    'bền':          ['chắc', 'chắc_chắn', 'bền_đẹp', 'chắc bền'],
    'mềm':          ['mịn', 'mềm_mại', 'êm', 'nhẹ'],
    'xinh':         ['đẹp', 'xịn', 'ưng', 'dễ thương', 'cute'],
    'ưng':          ['thích', 'oke', 'ok', 'hài_lòng', 'vừa ý'],
    'hoàn hảo':     ['tuyệt vời', 'tuyệt_vời', 'xuất sắc', 'quá đỉnh'],
    # Giao hàng
    'nhanh':        ['nhanh_chóng', 'siêu nhanh', 'lẹ', 'kịp thời', 'đúng hẹn'],
    'chậm':         ['trễ', 'lâu', 'muộn', 'chưa tới'],
    'đóng gói':     ['đóng_gói', 'bao bì', 'đóng hàng', 'gói hàng'],
    'cẩn thận':     ['kỹ', 'cẩn_thận', 'tỉ mỉ', 'chu đáo'],
    # Dịch vụ
    'nhiệt tình':   ['nhiệt_tình', 'tận tâm', 'chu đáo', 'tận_tình', 'vui vẻ'],
    'thân thiện':   ['thân_thiện', 'dễ chịu', 'hòa đồng', 'tốt bụng'],
    'hỗ trợ':       ['hỗ_trợ', 'tư vấn', 'giúp đỡ', 'hỗ trợ tốt'],
    'cảm ơn':       ['cảm_ơn', 'thanks', 'thank you', 'cảm ơn shop'],
    # Giá
    'rẻ':           ['hợp lý', 'hợp_lý', 'đáng tiền', 'giá tốt', 'phải chăng'],
    'đáng tiền':    ['xứng đáng', 'đáng mua', 'hợp lý', 'giá ổn'],
    'hợp lý':       ['hợp_lý', 'ổn', 'ok', 'phải chăng', 'đáng tiền'],
    # Recommend
    'nên mua':      ['đáng mua', 'mua đi', 'recommend', 'rcm', 'mua nha'],
    'recommend':    ['rcm', 'nên mua', 'giới thiệu', 'khuyến nghị'],
    # Mô tả
    'đúng mô tả':   ['đúng_mô_tả', 'chuẩn mô tả', 'như hình', 'giống hình', 'đúng như quảng cáo'],
    'giống hình':   ['đúng hình', 'như hình', 'y hình', 'chuẩn hình'],
    # Phàn nàn
    'tệ':           ['kém', 'chán', 'thất vọng', 'không ổn', 'không đạt'],
    'lỗi':          ['hỏng', 'bị lỗi', 'lỗi hàng', 'hàng lỗi', 'defect'],
    'mùi':          ['mùi nồng', 'nồng mùi', 'hôi', 'mùi khó chịu'],
}

print(f'Từ điển: {len(SYNONYMS)} từ gốc, trung bình {sum(len(v) for v in SYNONYMS.values())/len(SYNONYMS):.1f} synonym/từ')

Từ điển: 26 từ gốc, trung bình 4.6 synonym/từ


# **Cell 6 — Các hàm EDA**

In [ ]:
import random, re

def synonym_replace(text, n=1):
    '''Thay n từ bằng synonym ngẫu nhiên.'''
    words = text.split()
    new_words = words.copy()
    replaced = 0
    indices = list(range(len(words)))
    random.shuffle(indices)
    for i in indices:
        w = words[i].lower()
        if w in SYNONYMS and replaced < n:
            new_words[i] = random.choice(SYNONYMS[w])
            replaced += 1
    return ' '.join(new_words)

def random_swap(text, n=1):
    '''Đổi chỗ ngẫu nhiên n cặp từ.'''
    words = text.split()
    if len(words) < 3:
        return text
    new_words = words.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return ' '.join(new_words)

def random_delete(text, p=0.15):
    '''Xóa ngẫu nhiên mỗi từ với xác suất p.'''
    words = text.split()
    if len(words) <= 3:
        return text
    new_words = [w for w in words if random.random() > p]
    return ' '.join(new_words) if new_words else text

def random_insert(text, n=1):
    '''Chèn thêm n synonym ngẫu nhiên vào vị trí ngẫu nhiên.'''
    words = text.split()
    new_words = words.copy()
    all_synonyms = [s for syns in SYNONYMS.values() for s in syns]
    for _ in range(n):
        insert_word = random.choice(all_synonyms)
        insert_pos  = random.randint(0, len(new_words))
        new_words.insert(insert_pos, insert_word)
    return ' '.join(new_words)

def eda_augment(text, num_aug=4):
    '''
    Sinh num_aug biến thể từ 1 câu bằng cách kết hợp ngẫu nhiên 4 phép EDA.
    Trả về list các biến thể (khác câu gốc).
    '''
    augmented = set()
    ops = [synonym_replace, random_swap, random_delete, random_insert]
    attempts = 0
    while len(augmented) < num_aug and attempts < num_aug * 4:
        attempts += 1
        op  = random.choice(ops)
        new = op(text).strip()
        if new and new.lower() != text.lower() and len(new) > 3:
            augmented.add(new)
    return list(augmented)

# Quick test
test = 'chất lượng tốt giao hàng nhanh'
print('Câu gốc:', test)
print('EDA biến thể:')
for v in eda_augment(test, num_aug=4):
    print(' ', v)

Câu gốc: chất lượng tốt giao hàng nhanh
EDA biến thể:
  chất hàng tốt giao lượng nhanh
  chất lượng tốt giao nhanh hàng
  chất lượng chuẩn giao hàng nhanh
  chất lượng tốt giao hàng kịp thời


# **Cell 7 — Hàm Back-translation (Vi → En → Vi)**

In [ ]:
import time
from deep_translator import GoogleTranslator

def back_translate(text, sleep=SLEEP_BACKTRANS):
    '''
    Dịch Vi→En→Vi để tạo biến thể tự nhiên.
    sleep: thời gian chờ giữa 2 lần gọi (giảm nguy cơ bị Google chặn).
    '''
    try:
        en = GoogleTranslator(source='vi', target='en').translate(text)
        time.sleep(sleep)
        vi = GoogleTranslator(source='en', target='vi').translate(en)
        time.sleep(sleep)
        if vi and vi.lower().strip() != text.lower().strip():
            return vi.strip()
    except Exception as e:
        print(f'    [back_translate] Lỗi: {e}')
    return None

# Quick test (gọi 1 lần Google Translate)
sample = 'chất lượng tốt đúng như mô tả'
result = back_translate(sample)
print(f'Gốc  : {sample}')
print(f'Dịch : {result}')

Gốc  : chất lượng tốt đúng như mô tả
Dịch : None


#**Cell 8 — Hàm kiểm tra & loại trùng lặp**

In [ ]:
seen = set(df['text_clean'].str.strip().str.lower().tolist())

def deduplicate(existing, candidates):
    unique = []
    for c in candidates:
        c_norm = c.strip().lower()
        if c_norm and c_norm not in existing and len(c_norm) > 3:
            existing.add(c_norm)
            unique.append(c.strip())
    return unique

print(f'Đã load {len(seen)} câu gốc vào bộ kiểm tra trùng.')

Đã load 4527 câu gốc vào bộ kiểm tra trùng.


# **Cell 9 — Chạy augmentation (Back-translation + EDA)**

In [ ]:
import math

needed = compute_needed(df, TARGET_ROWS, MAX_PER_INTENT)

new_rows = []
grand_total_needed = sum(needed.values())
grand_collected = 0

for intent, n_needed in needed.items():
    examples = df[df['intent'] == intent]['text_clean'].tolist()
    random.shuffle(examples)

    n_bt  = math.ceil(n_needed * RATIO_BACKTRANS)  # Số câu từ back-translation
    n_eda = n_needed - n_bt                         # Số câu từ EDA

    print(f'\n▶ {intent} (cần +{n_needed}: BT={n_bt}, EDA={n_eda})')
    collected_bt  = 0
    collected_eda = 0

    # Phần 1: Back-translation
    bt_pool = (examples * (n_bt // max(len(examples),1) + 2))[:n_bt*3]
    for src in bt_pool:
        if collected_bt >= n_bt:
            break
        result = back_translate(src, sleep=SLEEP_BACKTRANS)
        if result:
            unique = deduplicate(seen, [result])
            if unique:
                new_rows.append({'text': unique[0], 'text_clean': unique[0], 'intent': intent})
                collected_bt += 1

    # Phần 2: EDA
    eda_pool = (examples * (n_eda // max(len(examples),1) + 2))[:n_eda*5]
    for src in eda_pool:
        if collected_eda >= n_eda:
            break
        variants = eda_augment(src, num_aug=4)
        unique   = deduplicate(seen, variants)
        for u in unique:
            if collected_eda >= n_eda:
                break
            new_rows.append({'text': u, 'text_clean': u, 'intent': intent})
            collected_eda += 1

    total_intent = collected_bt + collected_eda
    grand_collected += total_intent
    pct = grand_collected / grand_total_needed * 100
    print(f'  BT={collected_bt}, EDA={collected_eda} → tổng +{total_intent} | '
          f'tiến độ tổng: {grand_collected}/{grand_total_needed} ({pct:.0f}%)')

print('\nAugmentation hoàn thành!')


▶ khen_chat_luong (cần +18: BT=9, EDA=9)
  BT=9, EDA=9 → tổng +18 | tiến độ tổng: 18/7473 (0%)

▶ trung_tinh (cần +20: BT=10, EDA=10)
  BT=10, EDA=10 → tổng +20 | tiến độ tổng: 38/7473 (1%)

▶ dung_voi_mo_ta (cần +125: BT=63, EDA=62)
  BT=63, EDA=62 → tổng +125 | tiến độ tổng: 163/7473 (2%)

▶ phan_nan_chat_luong (cần +169: BT=85, EDA=84)
  BT=85, EDA=84 → tổng +169 | tiến độ tổng: 332/7473 (4%)

▶ khen_giao_hang (cần +173: BT=87, EDA=86)
  BT=87, EDA=86 → tổng +173 | tiến độ tổng: 505/7473 (7%)

▶ gia_tri_gia (cần +968: BT=484, EDA=484)
  BT=128, EDA=484 → tổng +612 | tiến độ tổng: 1117/7473 (15%)

▶ de_xuat_mua (cần +1500: BT=750, EDA=750)
  BT=67, EDA=750 → tổng +817 | tiến độ tổng: 1934/7473 (26%)

▶ phan_nan_giao_hang (cần +1500: BT=750, EDA=750)
  BT=65, EDA=750 → tổng +815 | tiến độ tổng: 2749/7473 (37%)

▶ khen_dich_vu (cần +1500: BT=750, EDA=750)
  BT=28, EDA=750 → tổng +778 | tiến độ tổng: 3527/7473 (47%)

▶ phan_nan_dich_vu (cần +1500: BT=750, EDA=750)
  BT=6, EDA=750 → tổn

#**Cell 10 — Lưu & tải xuống kết quả**

In [ ]:
from google.colab import files

df_new   = pd.DataFrame(new_rows)
df_final = pd.concat([df, df_new], ignore_index=True)
df_final = df_final.drop_duplicates(subset='text_clean').reset_index(drop=True)

df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f'Dòng ban đầu : {len(df)}')
print(f'Dòng mới sinh: {len(df_new)}')
print(f'Tổng sau aug : {len(df_final)}')
print(f'File lưu     : {OUTPUT_FILE}\n')
print('Phân bố intent sau augmentation:')
print(df_final['intent'].value_counts())

files.download(OUTPUT_FILE)

Dòng ban đầu : 4527
Dòng mới sinh: 4283
Tổng sau aug : 8810
File lưu     : data_augmented.csv

Phân bố intent sau augmentation:
intent
khen_chat_luong        1853
trung_tinh             1715
de_xuat_mua             903
phan_nan_giao_hang      885
khen_dich_vu            809
phan_nan_dich_vu        762
gia_tri_gia             755
dung_voi_mo_ta          394
khen_giao_hang          367
phan_nan_chat_luong     367
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>